In [3]:
!pip install requests beautifulsoup4 pandas

In [4]:
import requests
import pandas as pd
import time

from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

print("All Libraries Imported Successfully!")

All Libraries Imported Successfully!


In [5]:
base_url = "https://books.toscrape.com/"
robots_url = base_url + "robots.txt"

robots = RobotFileParser()
robots.set_url(robots_url)
robots.read()

print("Website:", base_url)
print("Can scrape homepage:",
      robots.can_fetch("*", base_url))

Website: https://books.toscrape.com/
Can scrape homepage: True


In [6]:
response = requests.get(
    base_url,
    headers={"User-Agent": "StudentBooksScraper/1.0"},
    timeout=15
)

print("Status Code:", response.status_code)
print("Website Connected Successfully!")

Status Code: 200
Website Connected Successfully!


In [7]:
robots_response = requests.get(robots_url, timeout=15)

robots = RobotFileParser()

if robots_response.status_code == 200:
    robots.parse(robots_response.text.splitlines())
elif robots_response.status_code == 404:
    robots.parse([])
    print("No robots.txt rules published.")
else:
    raise Exception("Could not verify robots.txt. Stop scraping.")

print("Can scrape:",
      robots.can_fetch("*", base_url))

No robots.txt rules published.
Can scrape: True


In [8]:
base_url = "https://books.toscrape.com/"

response = requests.get(
    base_url,
    headers={"User-Agent": "StudentBooksScraper/1.0"},
    timeout=15
)

print("Website URL:", base_url)
print("Status Code:", response.status_code)

if response.status_code == 200:
    print("Website Connected Successfully!")
else:
    print("Website Connection Failed!")

Website URL: https://books.toscrape.com/
Status Code: 200
Website Connected Successfully!


In [9]:
# Check robots.txt
robots_url = base_url + "robots.txt"

robots_response = requests.get(robots_url, timeout=15)

print("Robots.txt Status:", robots_response.status_code)

if robots_response.status_code == 200:
    robots = RobotFileParser()
    robots.parse(robots_response.text.splitlines())
    print("Robots.txt rules loaded.")
elif robots_response.status_code == 404:
    robots = RobotFileParser()
    robots.parse([])
    print("No published robots.txt rules found.")
else:
    print("Could not verify robots.txt. Stop here.")

Robots.txt Status: 404
No published robots.txt rules found.


In [10]:
soup = BeautifulSoup(response.text, "html.parser")

books = soup.find_all("article", class_="product_pod")

print("Total Books on First Page:", len(books))

first_book = books[0]

title = first_book.h3.a["title"]
price = first_book.find("p", class_="price_color").text.strip()
rating = first_book.find("p", class_="star-rating")["class"][1]
availability = first_book.find(
    "p", class_="instock availability"
).text.strip()

print("\nFirst Book Details")
print("Title:", title)
print("Price:", price)
print("Rating:", rating)
print("Availability:", availability)

Total Books on First Page: 20

First Book Details
Title: A Light in the Attic
Price: Â£51.77
Rating: Three
Availability: In stock


In [11]:
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

books = soup.find_all("article", class_="product_pod")

first_book = books[0]

price = first_book.find(
    "p", class_="price_color"
).text.strip()

print("Correct Price:", price)

Correct Price: £51.77


In [12]:
import requests
import pandas as pd
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin

base_url = "https://books.toscrape.com/"
page_url = base_url

all_books = []

session = requests.Session()
session.headers.update({
    "User-Agent": "StudentBooksScraper/1.0"
})

page_count = 0

while page_url:

    try:
        response = session.get(page_url, timeout=15)
        response.raise_for_status()

        response.encoding = "utf-8"

        soup = BeautifulSoup(response.text, "html.parser")

        books = soup.find_all(
            "article", class_="product_pod"
        )

        for book in books:

            try:
                title_tag = book.select_one("h3 a")
                price_tag = book.select_one(".price_color")
                rating_tag = book.select_one(".star-rating")
                availability_tag = book.select_one(
                    ".instock.availability"
                )

                title = (
                    title_tag.get("title", "").strip()
                    if title_tag else None
                )

                price_text = (
                    price_tag.get_text(strip=True)
                    if price_tag else None
                )

                price = (
                    float(price_text.replace("£", "").strip())
                    if price_text else None
                )

                rating = (
                    rating_tag.get("class", [])[-1]
                    if rating_tag else None
                )

                availability = (
                    availability_tag.get_text(" ", strip=True)
                    if availability_tag else None
                )

                product_url = (
                    urljoin(base_url, title_tag.get("href"))
                    if title_tag else None
                )

                all_books.append({
                    "Title": title,
                    "Price_GBP": price,
                    "Rating": rating,
                    "Availability": availability,
                    "Product_URL": product_url
                })

            except (ValueError, AttributeError, TypeError):
                continue

        page_count += 1

        print(
            "Page:", page_count,
            "| Books Collected:", len(all_books)
        )

        next_button = soup.select_one("li.next a")

        if next_button:
            page_url = urljoin(
                page_url,
                next_button.get("href")
            )
            time.sleep(1)
        else:
            page_url = None

    except requests.RequestException as e:
        print("Request Error:", e)
        break

print("\nScraping Completed!")
print("Total Pages:", page_count)
print("Total Books Collected:", len(all_books))

Page: 1 | Books Collected: 20
Page: 2 | Books Collected: 40
Page: 3 | Books Collected: 60
Page: 4 | Books Collected: 80
Page: 5 | Books Collected: 100
Page: 6 | Books Collected: 120
Page: 7 | Books Collected: 140
Page: 8 | Books Collected: 160
Page: 9 | Books Collected: 180
Page: 10 | Books Collected: 200
Page: 11 | Books Collected: 220
Page: 12 | Books Collected: 240
Page: 13 | Books Collected: 260
Page: 14 | Books Collected: 280
Page: 15 | Books Collected: 300
Page: 16 | Books Collected: 320
Page: 17 | Books Collected: 340
Page: 18 | Books Collected: 360
Page: 19 | Books Collected: 380
Page: 20 | Books Collected: 400
Page: 21 | Books Collected: 420
Page: 22 | Books Collected: 440
Page: 23 | Books Collected: 460
Page: 24 | Books Collected: 480
Page: 25 | Books Collected: 500
Page: 26 | Books Collected: 520
Page: 27 | Books Collected: 540
Page: 28 | Books Collected: 560
Page: 29 | Books Collected: 580
Page: 30 | Books Collected: 600
Page: 31 | Books Collected: 620
Page: 32 | Books Coll

In [13]:
df = pd.DataFrame(all_books)

print("Dataset Shape:", df.shape)

df.head(10)

Dataset Shape: (1000, 5)


,Title,Price_GBP,Rating,Availability,Product_URL
0,A Light in the Attic,51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,22.65,One,In stock,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,33.34,Four,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,17.93,Three,In stock,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,22.60,Four,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,52.15,One,In stock,https://books.toscrape.com/catalogue/the-black...


In [14]:
print("Missing Values Before Cleaning:")
print(df.isnull().sum())

df["Title"] = df["Title"].str.strip()
df["Rating"] = df["Rating"].str.strip()
df["Availability"] = df["Availability"].str.strip()

df = df.drop_duplicates(subset=["Product_URL"])

df = df.dropna(
    subset=["Title", "Price_GBP", "Rating"]
)

df["Price_GBP"] = df["Price_GBP"].astype(float)

print("\nDataset Shape After Cleaning:", df.shape)

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

Missing Values Before Cleaning:
Title           0
Price_GBP       0
Rating          0
Availability    0
Product_URL     0
dtype: int64

Dataset Shape After Cleaning: (1000, 5)

Missing Values After Cleaning:
Title           0
Price_GBP       0
Rating          0
Availability    0
Product_URL     0
dtype: int64


In [15]:
df.to_csv(
    "books_scraped_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV File Saved Successfully!")

CSV File Saved Successfully!


In [16]:
test_df = pd.read_csv("books_scraped_data.csv")

print("CSV Loaded Successfully!")
print("Total Records:", len(test_df))

test_df.head()

CSV Loaded Successfully!
Total Records: 1000


,Title,Price_GBP,Rating,Availability,Product_URL
0,A Light in the Attic,51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...


In [17]:
print("Dataset Shape:", df.shape)

print("Total Books:", len(df))

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (1000, 5)
Total Books: 1000

Column Names:
['Title', 'Price_GBP', 'Rating', 'Availability', 'Product_URL']

Missing Values:
Title           0
Price_GBP       0
Rating          0
Availability    0
Product_URL     0
dtype: int64


In [18]:
df["Title"] = df["Title"].str.strip()

df["Rating"] = df["Rating"].str.strip()

df["Availability"] = df["Availability"].str.strip()

df = df.drop_duplicates(subset=["Product_URL"])

df = df.dropna(
    subset=["Title", "Price_GBP", "Rating"]
)

df["Price_GBP"] = df["Price_GBP"].astype(float)

print("Data Cleaning Completed!")
print("Final Dataset Shape:", df.shape)

df.head()

Data Cleaning Completed!
Final Dataset Shape: (1000, 5)


,Title,Price_GBP,Rating,Availability,Product_URL
0,A Light in the Attic,51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...


In [19]:
df.to_csv(
    "books_scraped_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV File Saved Successfully!")

CSV File Saved Successfully!


In [20]:
test_df = pd.read_csv("books_scraped_data.csv")

print("CSV Loaded Successfully!")
print("Total Records:", len(test_df))

test_df.head()

CSV Loaded Successfully!
Total Records: 1000


,Title,Price_GBP,Rating,Availability,Product_URL
0,A Light in the Attic,51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...
